In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

### Generating Tables 

This creates the tables of the accuracies and clustering scores for distinguishing parameter choices of the computed distributions. Note that you first have to run `sim_params.py`.

In [ ]:
# read in all results and configurations
resdir = 'simulations/results/sim_params'
res_df = pd.read_csv(f'{resdir}/results_combined.csv', header=[0,1], index_col=0)

configs = pd.read_csv(f'{resdir}/configs.csv', header=[0], index_col=0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_agglo', 'ari_kmeans']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

# group methods
method_group = {}
for method in list(res_df.index):
    if 'toper' in method:
        method_group[method] = 'TopER'
    elif 'kernel' in method:
        method_group[method] = 'Kernel'
    elif 'embedding' in method or 'spectral_zoo' == method:
        method_group[method] = 'Embeddings'
    elif 'divcurves' in method:
        method_group[method] = 'Diversity Curves'
    elif 'div_function' in method:
        method_group[method] = 'Diversity Function'
    elif 'persistent_homology' in method:
        method_group[method] = 'Persistent Homology'

res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
        'divcurves_shortest_path_distance': 'Shortest Path',
        'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
        'div_function_shortest_path_distance': 'Profile Shortest Path', 
        'graph_embedding_netLSD': 'NetLSD',
        'graph_embedding_graph2vec': 'Graph2Vec', 
        'graph_embedding_feathergraph': 'FEATHER',
        'kernel_shortest_path': 'Kernel Shortest Path', 
        'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
        'kernel_pyramid_match': 'Kernel Pyramid Match', 
        'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
        'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
        'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
        'kernel_weisfeiler_lehman': 'Kernel WL',
        'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
        'kernel_core_framework': 'Kernel Core Framework',
        'kernel_mds_core_framework': 'Kernel Core Framework MDS',
        'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
        'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
        'toper_degree_sub': 'TopER Degree', 
        'toper_popularity_sub': 'TopER Popularity', 
        'toper_forricci_sub': 'TopER Forricci',
        'toper_olricci_sub': 'TopER Olricci',
        'toper_closeness_sub': 'TopER Closeness', 
        'toper_all_sub': 'TopER All Filtrations',
        'spectral_zoo': 'Spectral Zoo',
        'persistent_homology_degree': 'PH Degree',
        'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
        'persistent_homology_forman_ricci_curvature': 'PH Fricci',
        'persistent_homology_laplacian': 'PH Laplacian', 
        'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
        'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)


# create table for Latex
results_latex_dfs = {}
models = list(np.unique(df.columns.get_level_values(0)))

for model in models:
    results_latex_model = {}
    # sort wrt to accuracy, in groups
    df_model = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values((model, 'acc_mean'), ascending=False)).droplevel(0)
    res_copy = df_model.copy()
    for score in scores:
        mean_col = (model, f'{score}_mean')
        std_col = (model, f'{score}_std')
        if mean_col in res_copy.columns and std_col in res_copy.columns:
            # formatting of the latex table
            results_latex_model[score] = res_copy[mean_col].combine(
                res_copy[std_col],
                lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
            )
    results_latex_df_model = pd.DataFrame.from_dict(results_latex_model)

# Bold the best mean for each score
    for score in scores:
        mean_col = (model, f"{score}_mean")
        display_col = (model, score)
        if mean_col in res_copy.columns:
            if not scores_asc[score]: 
                best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
            else:
                best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
            results_latex_df_model.loc[best_mask, score] = results_latex_df_model.loc[best_mask, score].apply(
                lambda x: f"\\textbf{{{x}}}"
            )
        else:
            print(f'{mean_col} not in dataframe')

    #final adjustments latex df 
    results_latex_df_model.index.rename([None, 'Method'], inplace = True)
    new_cols = {
        'acc': 'Accuracy', 
        'silh': 'Silhouette Score', 
        'db': 'Davies-Bouldin', 
        'ch': 'Calinski-Harabasz', 
        'ari_kmeans': 'ARI k-means', 
        'ari_agglo': 'ARI agglomerative'
    }

    results_latex_df_model = results_latex_df_model.rename(columns=new_cols)
    results_latex_df_model.replace(to_replace='nan $\pm $ nan', value='-', inplace=True)

    #store df in dict
    results_latex_dfs[model] = results_latex_df_model

# print table for each model
for model in models:
    print(f'model: {model} with config' + configs.loc[(model, 'configurations')])
    print(results_latex_dfs[model].to_latex(multirow=False, escape = False))

### Generating cluster score plot

Here we create the plot of the silhouette score for the selected methods. Note that you first have to run `sim_params.py`.

In [ ]:
#loading results and configuration
resdir = 'simulations/results/sim_params'
res_df = pd.read_csv(f'{resdir}/results_combined.csv', header=[0,1], index_col=0)
configs = pd.read_csv(f'{resdir}/configs.csv', header=[0], index_col=0)
# selecting the methods we want to plot
methods_selection = [
    'divcurves_shortest_path_distance',
    'kernel_shortest_path',
    'kernel_weisfeiler_lehman_optimal_assignment',
    'toper_all_sub',
    'graph_embedding_netLSD',
    'graph_embedding_feathergraph',
    'spectral_zoo',
    'persistent_homology_vietoris_rips'
]

#  renaming
names_methods = {
    'divcurves_pca_shortest_path_distance': 'Diversity Curves',
    'divcurves_shortest_path_distance': 'Diversity Curves',
    'graph_embedding_feathergraph': 'FEATHER',
    'kernel_shortest_path': 'Kernel Shortest Path',
    'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Opt. Assig.',
    'toper_forricci_sub': 'TopER Forricci',
    'spectral_zoo': 'Spectral Zoo',
    'persistent_homology_alpha_complex': 'PH Alpha Complex',
    'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
    'toper_all_sub': 'TopER',
    'graph_embedding_netLSD': 'NetLSD',
}

models = ['ER', 'RPG', 'RG', 'SBM']

# selecting score to plot
score = 'silh'

#select relevant methods to plot
selected = res_df.loc[methods_selection,(models, slice(None))]

y=selected.loc[:, (slice(None), [f'{score}_mean'])].droplevel(level=1, axis=1)
y_err = selected.loc[:, (slice(None), [f'{score}_std'])].droplevel(level=1, axis=1)



fig, ax = plt.subplots(figsize=(10, 6))

# Get the models (column names)
# models = y.columns  
n_rows = len(y)
x_positions = np.arange(len(models))

# Width for offsetting errorbars within each model group
bar_width = 0.5 / n_rows  

# Colors for each row and font size 
colors = plt.cm.Dark2(X=range(n_rows))
matplotlib.rcParams.update({'font.size': 20})


bars = {}

# Plot errorbars for each row
for row_idx, row_name in enumerate(y.index):
    offsets = x_positions + (row_idx - n_rows/2 + 0.5) * bar_width
    ax.errorbar(
        offsets, 
        y.iloc[row_idx],
        yerr=y_err.iloc[row_idx],
        fmt='none',
        label=row_name,
        color='black',
        capsize=3,
        markersize=6
    )
    bars[row_name] = ax.bar(offsets, y.iloc[row_idx], width=bar_width, color=colors[row_idx])

ax = plt.gca()
ax.set_xticks(x_positions)
ax.set_xticklabels(models)
ax.set_ylabel('Silhouette Score')
ax.set_ylim(top=1) 
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(False)
box = ax.get_position()

ax.legend(handles = [bars[row] for row in y.index],
           labels=[names_methods[row] for row in y.index],
           frameon=False,
           loc=(0.62, 0.45),
           )


ax.tick_params(axis='x', which='both', bottom=False, top=False, left=False, right=False, labelbottom=True, labelleft=False)

plt.show()